## Orchestrating Agents as Tools

Here is the clean, properly formatted Markdown version of your text:

# Orchestrating Agents as Tools

Welcome back! In the previous lesson, you mastered building agentic pipelines in which specialized agents work together in a fixed sequence. Today, we're taking a significant architectural leap forward by learning how to build agent orchestration systems, where a central planner agent can dynamically decide which specialist agents to call based on the specific task at hand.

The key insight is that complex agentic systems can be wrapped as simple tools for other agents. This creates a powerful hierarchy in which agents can leverage the full capabilities of other agents as easily as they use basic functions, enabling much more flexible and intelligent problem-solving than fixed pipeline sequences.

## The Agent-as-Tool Concept

The fundamental difference between pipelines and orchestration lies in decision-making authority. In a pipeline, you as the developer decide the sequence: every problem goes through analysis, then calculation, then presentation. In orchestration, the central agent makes these decisions dynamically based on the nature of each specific request. The key insight is that we can encapsulate an agent call as a tool, allowing an orchestrator agent with access to that tool to seamlessly call another specialist agent.

Here's how the agent orchestration flow works:

1. User submits a request to the orchestrator agent
2. Orchestrator analyzes the request and decides whether to handle it directly or delegate to a specialist
3. If delegation is needed, the orchestrator calls the specialist agent as a tool
4. Specialist processes the request and returns results to the orchestrator
5. Orchestrator provides the final response to the user

Consider how this changes the system's behavior. If someone asks, *"What is the capital of France?"*, a pipeline system would still run the question through all three agents unnecessarily. An orchestrated system, however, allows the central agent to recognize that this is a straightforward knowledge question requiring no mathematical tools and respond directly. But when someone asks about solving equations, the orchestrator can intelligently delegate this to a calculator specialist. This dynamic delegation creates systems that are both more efficient and more capable.

## Building Specialist Agents

Before we can orchestrate agents, we need to create the specialist agents that will serve as tools. Let's build a calculator assistant that specializes in mathematical problem-solving, designed to work as a standalone tool that can handle mathematical questions from start to finish.

```python
import json
from agent import Agent
from functions import sum_numbers, multiply_numbers, subtract_numbers, divide_numbers, power, square_root

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

```

This calculator assistant is designed as a complete mathematical problem-solver with access to all the mathematical tools. The system prompt establishes the agent's expertise in mathematics while allowing it the flexibility to handle various types of mathematical problems. This balance is important for specialist agents — they need clear domain focus while maintaining enough flexibility to be useful as tools for different orchestrators.

## Creating Agent Tool Wrappers

To enable agents to use other agents as tools, we need to wrap any agent into a callable function that follows the same interface as our other tools. This helper function will streamline our workflow: we'll call it with our specialist agent to get back both a callable function and its schema, which we can then pass to our orchestrator agent, enabling it to call the specialist agent just like any other tool.

```python
def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.
    
    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does
    
    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        print(f"🦾 Agent tool called ({agent.name}): {message}")
        
        # Call the agent
        _, response = agent.run([{"role": "user", "content": message}])
        
        print(f"📊 Agent response ({agent.name}): {response}")
        
        return response
    
    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }
    
    return agent_tool_function, tool_schema

```

The key part of this function is the inner `agent_tool_function`. When we call `agent.run()`, it returns two things: the complete message history and the final text response. We use `_` to ignore the message history because when one agent calls another as a tool, it only needs the final answer — not the entire conversation history with all the intermediate function calls and reasoning steps.

For example, if our orchestrator asks the calculator agent to solve an equation, it just wants the final answer like `"x = 3 and x = 2"`, not the detailed trace of every mathematical function that was called along the way. This keeps the tool interface clean and focused on results.

## Implementing the Orchestrator Agent

With our specialist agent ready and our wrapper function defined, we can now create the agent tool wrapper and build our orchestrator agent. The orchestrator will be a general-purpose assistant that can handle various types of questions, using the calculator assistant when mathematical expertise is needed.

```python
# Create agent tool for calculator
calculator_tool_function, calculator_tool_schema = create_agent_tool(
    calculator_assistant,
    "Call the calculator assistant to solve mathematical problems and equations."
)

# Create a general assistant
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and use the calculator tool for math problems.",
    tools = {calculator_tool_schema["name"]: calculator_tool_function},
    tool_schemas=[calculator_tool_schema]
)

```

The orchestrator agent (`helpful_assistant`) has a deliberately general system prompt that doesn't limit it to any specific domain. Instead, it's designed to be helpful across various tasks while knowing it has access to mathematical expertise through the calculator tool. Notice how we pass the calculator tool to the orchestrator just like any other tool — from the orchestrator's perspective, calling another agent is no different from calling a mathematical function.

## Testing General Knowledge Questions

Let's test our orchestrated system with a general knowledge question to see how the orchestrator handles tasks that don't require specialist agents.

```python
# Create a message list with a general knowledge question
messages = [
    {
        'role': 'user', 
        'content': 'What is the capital of France?'
    }
]

# Run the orchestrator agent
result_messages, response = helpful_assistant.run(messages)

# Display the orchestrator's response to the user
print(response)

```

When we run this code, the orchestrator agent will receive the question and analyze whether it needs specialist assistance. Since this is a straightforward factual question about geography, the agent should recognize that it can provide the answer directly from its general knowledge without calling any tools.

Here's what happens when we execute this request:

```text
The capital of France is Paris. Paris is the largest city in France and serves as the country's political, economic, and cultural center. It's located in the north-central part of France along the Seine River and is known for iconic landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum.

```

The orchestrator correctly identified this as a question it could handle directly using its general knowledge without calling any tools. The response was immediate and appropriate, demonstrating the system's efficiency for simple tasks that don't require specialist assistance.

## Testing Mathematical Problem Delegation

Now let's test the system with a mathematical question that requires the specialist agent's expertise.

```python
# Create a message list with a general knowledge question
messages = [
    {
        'role': 'user', 
        'content': 'What is the solution to the equation x² - 5x + 6 = 0?'
    }
]

# Run the orchestrator agent
result_messages, response = helpful_assistant.run(messages)

# Display the orchestrator's response to the user
print('\n=== Final Response ===\n')
print(response)

```

In this example, the orchestrator will recognize that solving a quadratic equation requires mathematical expertise beyond basic knowledge. The agent should automatically decide to delegate this task to the calculator assistant tool, which will then use its mathematical functions to solve the equation step by step.

Here's the complete execution trace showing the delegation and calculation process:

```text
🔧 Tool called: calculator_assistant_agent({'message': 'Solve the quadratic equation x² - 5x + 6 = 0'})
🦾 Agent tool called (calculator_assistant): Solve the quadratic equation x² - 5x + 6 = 0
🔧 Tool called: power({'base': -5, 'exponent': 2})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 1})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 6})
🔧 Tool called: subtract_numbers({'a': 25, 'b': 24})
🔧 Tool called: square_root({'number': 1})
🔧 Tool called: sum_numbers({'a': 5, 'b': 1})
🔧 Tool called: divide_numbers({'a': 6, 'b': 2})
🔧 Tool called: subtract_numbers({'a': 5, 'b': 1})
🔧 Tool called: divide_numbers({'a': 4, 'b': 2})
📊 Agent response (calculator_assistant): The solutions to the quadratic equation x² - 5x + 6 = 0 are:

**x = 3** and **x = 2**

This can be verified by factoring: x² - 5x + 6 = (x - 3)(x - 2) = 0

Therefore, x = 3 or x = 2.

=== Final Response ===

The solution to the equation x² - 5x + 6 = 0 is:

**x = 3** and **x = 2**

This quadratic equation can be solved by factoring. The expression x² - 5x + 6 factors into (x - 3)(x - 2) = 0, which gives us the two solutions where each factor equals zero:
- x - 3 = 0, so x = 3
- x - 2 = 0, so x = 2

You can verify these solutions by substituting them back into the original equation:
- For x = 3: (3)² - 5(3) + 6 = 9 - 15 + 6 = 0 ✓
- For x = 2: (2)² - 5(2) + 6 = 4 - 10 + 6 = 0 ✓

```

For the mathematical equation, the orchestrator recognized that this required mathematical expertise and delegated the task to the calculator assistant. The debug output shows the agent tool being called with the original question, the calculator assistant processing the request and providing a complete mathematical solution with factoring, explanation, and verification.

The orchestrator then received this detailed response from the specialist agent and presented the final answer to the user, demonstrating the full delegation and response cycle.

## Summary & Exercises

You've now learned how to build agent orchestration systems in which a central planner can dynamically delegate tasks to specialist agents. This architectural pattern offers significant advantages over fixed pipelines: it's more efficient for simple tasks, more flexible for complex problems, and allows you to build increasingly sophisticated specialist agents without complicating the overall system design.

In your upcoming practice exercises, you'll build your own orchestrated agent systems with multiple specialists for different domains. This foundation will enable you to create much more sophisticated and flexible agentic systems that can adapt their approach based on the specific requirements of each task.

## Creating Agent Tool Schemas

Now that you've seen how agent orchestration works with the complete create_agent_tool function, let's build a simplified version that focuses on just the schema creation part.

Your task is to implement a create_agent_schema function that generates a proper Claude tool schema for wrapping an agent as a tool. This function should:

    Accept two parameters: an agent and a description
    Create and return a tool schema dictionary with the correct structure
    Use the agent's name with _agent appended as the schema name
    Include the provided description in the schema
    Define an input schema with a message property of type string

After implementing your function, test it by calling it with the calculator_assistant and an appropriate description, then print the resulting schema using using a properly formatted output.

```
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# TODO: Define the create_agent_schema function that takes an agent and description as parameters
# The function should create and return a tool schema dictionary with:
# - A "name" field using the pattern: agent.name + "_agent"
# - A "description" field using the provided description parameter
# - An "input_schema" field with the proper structure for a message parameter


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Call your create_agent_schema function with the calculator_assistant and an appropriate description

# TODO: Print the resulting schema using a readable output
```

Here is the complete Python code implementing the `create_agent_schema` function, followed by creating the schema for the `calculator_assistant` and printing it in a cleanly formatted JSON structure.

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# TODO: Define the create_agent_schema function that takes an agent and description as parameters
def create_agent_schema(agent, description):
    """
    Generates a Claude-compatible tool schema for an agent wrapper.
    """
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message or task description to send to the specialist agent"
                }
            },
            "required": ["message"]
        }
    }
    return tool_schema


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Call your create_agent_schema function with the calculator_assistant and an appropriate description
calculator_description = "Call the calculator assistant to break down and solve mathematical problems and equations."
calculator_schema = create_agent_schema(calculator_assistant, calculator_description)

# TODO: Print the resulting schema using a readable output
print("================================================================================")
print("GENERATED AGENT TOOL SCHEMA")
print("================================================================================")
print(json.dumps(calculator_schema, indent=4))
print("================================================================================")

```

## Building Complete Agent Tool Wrappers

Great work on creating agent tool schemas! Now it's time to bring those schemas to life by building the complete wrapper function that makes agents callable as tools.

The key addition is defining an inner function called agent_tool_function inside your main function. This inner function should:

    Accept a message parameter
    Print a debug message showing the agent name and incoming message
    Call agent.run() with the message formatted as a proper message list
    Use tuple unpacking with _ to ignore the message history and capture only the response
    Print a debug message showing the agent name and response
    Return the response

Once your function is complete, test it by calling create_agent_tool with the calculator assistant and capturing both returned values. Then, call the tool function directly with a math question and print the result to see your agent wrapper in action!

```
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_schema(agent, description):
    
    # TODO: Define an inner function called agent_tool_function that accepts a message parameter
    # This function should:
    # - Print a debug message showing the agent name and incoming message
    # - Call agent.run() with a properly formatted message list: [{"role": "user", "content": message}]
    # - Use tuple unpacking with _ to ignore the message history and capture only the response
    # - Print a debug message showing the agent name and response
    # - Return the response
    
    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }
    
    # TODO: Return a tuple containing both the agent_tool_function and the tool_schema
    return tool_schema


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Call create_agent_tool (not create_agent_schema) with the calculator_assistant and description
# Use tuple unpacking to capture both the tool function and the schema

# TODO: Test the tool function directly by calling it with a math question like "What is 25 multiplied by 4?"

# TODO: Print the returned result
```

Here is the complete Python code implementing the requested inner executable function, the tuple return format, and the invocation to test the wrapper directly:

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    
    # TODO: Define an inner function called agent_tool_function that accepts a message parameter
    def agent_tool_function(message):
        # - Print a debug message showing the agent name and incoming message
        print(f"🦾 Agent tool called ({agent.name}): {message}")
        
        # - Call agent.run() with a properly formatted message list: [{"role": "user", "content": message}]
        # - Use tuple unpacking with _ to ignore the message history and capture only the response
        _, response = agent.run([{"role": "user", "content": message}])
        
        # - Print a debug message showing the agent name and response
        print(f"📊 Agent response ({agent.name}): {response}")
        
        # - Return the response
        return response
    
    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }
    
    # TODO: Return a tuple containing both the agent_tool_function and the tool_schema
    return agent_tool_function, tool_schema


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Call create_agent_tool (not create_agent_schema) with the calculator_assistant and description
# Use tuple unpacking to capture both the tool function and the schema
calculator_description = "Call the calculator assistant to solve mathematical problems and equations."
calc_tool_func, calc_tool_schema = create_agent_tool(calculator_assistant, calculator_description)

# TODO: Test the tool function directly by calling it with a math question like "What is 25 multiplied by 4?"
print("--- Testing Agent Tool Wrapper Directly ---")
test_message = "What is 25 multiplied by 4?"
result = calc_tool_func(test_message)

# TODO: Print the returned result
print("\n=== Final Returned Result ===")
print(result)

```

## Building Your Orchestrator Agent

Excellent work building the agent tool wrapper components! Now it's time to see the full orchestration system in action by creating an orchestrator agent that can intelligently decide when to delegate tasks to your calculator specialist.

Your task is to:

    Create the orchestrator agent called helpful_assistant with:
        A general system prompt explaining that it can assist with various tasks and use the calculator tool for math problems
        A tools dictionary containing the calculator tool function mapped to its schema name
        A tool_schemas list containing the calculator tool schema
    Next, test your orchestration system with two different scenarios.
        First, test it with general knowledge question to see it handle the question directly.
        Then, test it with a math question to see if it uses the specialist agent as a tool.

Watch closely as your orchestrator makes intelligent decisions about when to use its specialist agent versus handling questions on its own!

```
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.
    
    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does
    
    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        print(f"🦾 Agent tool called ({agent.name}): {message}")
        
        # Call the agent
        _, response = agent.run([{"role": "user", "content": message}])
        
        print(f"📊 Agent response ({agent.name}): {response}")
        
        return response
    
    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }
    
    return agent_tool_function, tool_schema


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create agent tool for calculator
calculator_tool_function, calculator_tool_schema = create_agent_tool(
    calculator_assistant,
    "Call the calculator assistant to solve mathematical problems and equations."
)

# TODO: Create the helpful_assistant orchestrator agent with:
# - name set to "helpful_assistant"
# - A general system prompt explaining it can assist with various tasks and use the calculator tool for math problems
# - tools dictionary containing the calculator tool function mapped to its schema name
# - tool_schemas list containing the calculator tool schema

# TODO: Create a messages list with a general knowledge question (e.g., "What is the capital of France?")

# TODO: Run the helpful_assistant with the messages list and capture both result_messages and response

# TODO: Print the response

# TODO: Create a new messages list with a math question (e.g., "What is the solution to the equation x² - 5x + 6 = 0?")

# TODO: Run the helpful_assistant with the new messages list and capture both result_messages and response

# TODO: Print a section header like "=== Final Response ===" followed by the response
```

Here is the full completed Python code implementing the orchestrator agent (`helpful_assistant`), mapped to the compiled agent tool wrapper function and schema. It then tests both a general knowledge question (handled directly) and a math equation question (delegated to the specialist agent).

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.
    
    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does
    
    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        print(f"🦾 Agent tool called ({agent.name}): {message}")
        
        # Call the agent
        _, response = agent.run([{"role": "user", "content": message}])
        
        print(f"📊 Agent response ({agent.name}): {response}")
        
        return response
    
    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }
    
    return agent_tool_function, tool_schema


# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create agent tool for calculator
calculator_tool_function, calculator_tool_schema = create_agent_tool(
    calculator_assistant,
    "Call the calculator assistant to solve mathematical problems and equations."
)

# TODO: Create the helpful_assistant orchestrator agent with:
# - name set to "helpful_assistant"
# - A general system prompt explaining it can assist with various tasks and use the calculator tool for math problems
# - tools dictionary containing the calculator tool function mapped to its schema name
# - tool_schemas list containing the calculator tool schema
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and use the calculator tool for math problems.",
    tools={calculator_tool_schema["name"]: calculator_tool_function},
    tool_schemas=[calculator_tool_schema]
)

# --- Test Scenario 1: General Knowledge Question ---
print("--- Scenario 1: Testing General Knowledge ---")

# TODO: Create a messages list with a general knowledge question (e.g., "What is the capital of France?")
messages_gk = [
    {
        'role': 'user', 
        'content': 'What is the capital of France?'
    }
]

# TODO: Run the helpful_assistant with the messages list and capture both result_messages and response
result_messages_gk, response_gk = helpful_assistant.run(messages_gk)

# TODO: Print the response
print(response_gk)


# --- Test Scenario 2: Mathematical Problem Delegation ---
print("\n--- Scenario 2: Testing Mathematical Delegation ---")

# TODO: Create a new messages list with a math question (e.g., "What is the solution to the equation x² - 5x + 6 = 0?")
messages_math = [
    {
        'role': 'user', 
        'content': 'What is the solution to the equation x² - 5x + 6 = 0?'
    }
]

# TODO: Run the helpful_assistant with the new messages list and capture both result_messages and response
result_messages_math, response_math = helpful_assistant.run(messages_math)

# TODO: Print a section header like "=== Final Response ===" followed by the response
print('\n=== Final Response ===\n')
print(response_math)

```

## Adding a Verification Specialist Agent